# PyTorch: ViT using pre-trained weights

In [1]:
import torch
import random
from torchinfo import summary
from torch import nn
from torch import optim
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torchmetrics.classification import MulticlassAccuracy
from common import CV_DATASETS_DIR
from common.torch import train, get_summary_writer, get_default_device, set_seed

2026-07-06 18:41:12.037346: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-06 18:41:12.062976: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
set_seed()
device = get_default_device()

In [3]:
print(f"PyTorch: version {torch.__version__}")
print(f"PyTorch: {device.type.upper()} device")

PyTorch: version 2.11.0+cu126
PyTorch: CUDA device


In [4]:
# Hyperparameters
BATCH_SIZE = 32
N_EPOCHS = 5
# Other parameters
IMAGE_SIZE = (224,224)

## Prepare Datasets

In [5]:
DATASET_PATH = CV_DATASETS_DIR/"animals"

tr_dataset = datasets.OxfordIIITPet(root=DATASET_PATH,
                                    transform=ViT_B_16_Weights.DEFAULT.transforms(),
                                    split="trainval",
                                    download=True)
ts_dataset = datasets.OxfordIIITPet(root=DATASET_PATH,
                                    transform=ViT_B_16_Weights.DEFAULT.transforms(),
                                    split="test",
                                    download=True)

assert tr_dataset.class_to_idx == ts_dataset.class_to_idx, "Train and Test class indices mismatch!"
len(tr_dataset), len(ts_dataset)

(3680, 3669)

In [6]:
tr_dl = DataLoader(tr_dataset, batch_size=BATCH_SIZE, num_workers=2, shuffle=True)
ts_dl = DataLoader(ts_dataset, batch_size=BATCH_SIZE, num_workers=2)
len(tr_dl), len(ts_dl)

(115, 115)

In [7]:
n_classes = len(tr_dataset.classes)
n_classes

37

## Define Model

In [8]:
# Create a model using pretrained weights
model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT).to(device)

for parameter in model.parameters():
    parameter.requires_grad = False

model.heads = nn.Sequential(
    nn.Linear(model.heads[0].in_features, n_classes)
).to(device)

In [ ]:
summary(model=model,
        input_size=(1, 3)+IMAGE_SIZE,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

## Train Model

In [9]:
logs_dir, writer = get_summary_writer("pt_vit_transfer_learning", "vit_with_weights", "5_epochs")
logs_dir

'runs/2026-07-06/pt_vit_transfer_learning/vit_with_weights/5_epochs'

In [10]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), weight_decay=0.1)
accuracy = MulticlassAccuracy(num_classes=n_classes).to(device)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

In [11]:
train(model, tr_dl, ts_dl, optimizer, criterion, accuracy, N_EPOCHS, writer, device, scheduler)

> Epoch: 01


  0%|          | 0/115 [00:00<?, ?it/s]

  0%|          | 0/115 [00:00<?, ?it/s]

Train L/A: 1.404/0.843 Test L/A: 1.021/0.879

> Epoch: 02


  0%|          | 0/115 [00:00<?, ?it/s]

...

  0%|          | 0/115 [00:00<?, ?it/s]

Train L/A: 0.947/0.958 Test L/A: 1.000/0.940

> Epoch: 05


  0%|          | 0/115 [00:00<?, ?it/s]

  0%|          | 0/115 [00:00<?, ?it/s]

Train L/A: 0.936/0.962 Test L/A: 1.000/0.944

